# LSTM Modeling v3 — Log-Transform + Arsitektur Simpel
### Notebook 03 v3 — 13 Fitur, log1p target, LSTM(64→32), Window=12

## Cell 1 — Import & Load

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

tf.random.set_seed(42)
np.random.seed(42)
print(f'TensorFlow : {tf.__version__}')

WORKDIR = Path(".")
INPUT_CSV = WORKDIR / 'indonesia_earthquakes_clustered.csv'
SUMM_CSV  = WORKDIR / 'output_spatio_temporal' / 'spatio_temporal_summary.csv'
GAP_CSV   = WORKDIR / 'output_spatio_temporal' / 'seismic_gap_zones.csv'
OUTDIR    = WORKDIR / 'output_lstm'
MODELDIR  = WORKDIR / 'models'
OUTDIR.mkdir(exist_ok=True)
MODELDIR.mkdir(exist_ok=True)

df = pd.read_csv(INPUT_CSV)
df['time'] = pd.to_datetime(df['time'], format='mixed', errors='coerce')
df['year'] = df['time'].dt.year

ZONES = ['Zona Sumatera', 'Zona Jawa-Bali-NTB', 'Zona Sulawesi-NTT',
         'Zona Maluku', 'Zona Papua']
COLORS = {
    'Zona Sumatera':      '#e41a1c',
    'Zona Jawa-Bali-NTB': '#377eb8',
    'Zona Sulawesi-NTT':  '#4daf4a',
    'Zona Maluku':        '#ff7f00',
    'Zona Papua':         '#984ea3',
}

print(f'Dataset : {len(df):,} baris x {len(df.columns)} kolom')
print(f'Rentang : {df["time"].min().date()} - {df["time"].max().date()}')
print()
print(f'{"Zona":<25} {"N Events":>9} {"%":>8}')
print('-' * 46)
for z in ZONES:
    n = (df['zone_name'] == z).sum()
    print(f'  {z:<23} {n:>9,} {n/len(df)*100:>7.1f}%')
print(f'  {"TOTAL":<23} {len(df):>9,} {"100.0%":>8}')


## Cell 2 — Feature Engineering (13 Fitur + log1p pada count/energy)

In [ ]:
def calc_energy(mags):
    return float(np.sum(10 ** (1.5 * np.asarray(mags, dtype=float) + 4.8)))

def mean_inter_event(times):
    t = sorted(times.dropna())
    if len(t) < 2:
        return 0.0
    diffs = [(t[i+1] - t[i]).total_seconds() / 86400.0 for i in range(len(t)-1)]
    return float(np.mean(diffs))

# 13 fitur — urutan HARUS konsisten (col 0 = target)
FEATURES = [
    'monthly_count',     # 0  target (log1p)
    'monthly_mean_mag',  # 1
    'monthly_max_mag',   # 2
    'monthly_energy',    # 3  (log1p)
    'inter_event_time',  # 4
    'rolling_mean_3',    # 5
    'rolling_mean_6',    # 6
    'rolling_std_3',     # 7
    'cumulative_energy', # 8  (log1p)
    'lag_1',             # 9
    'lag_2',             # 10
    'lag_3',             # 11
    'mag_anomaly',       # 12
]
N_FEATURES = len(FEATURES)  # 13
# kolom yang akan di-log1p sebelum scaling
LOG_COLS = ['monthly_count', 'monthly_energy', 'cumulative_energy']

date_range = pd.period_range('1950-01', '2026-05', freq='M')

ts_all   = {}  # raw (original scale) untuk evaluasi & plot
ts_log   = {}  # setelah log1p transform — dipakai untuk training

for z in ZONES:
    sub       = df[df['zone_name'] == z].copy()
    sub['ym'] = sub['time'].dt.to_period('M')

    grp = sub.groupby('ym').agg(
        monthly_count    = ('mag', 'count'),
        monthly_mean_mag = ('mag', 'mean'),
        monthly_max_mag  = ('mag', 'max'),
    )
    energy_s = sub.groupby('ym')['mag'].apply(calc_energy).rename('monthly_energy')
    inter_s  = sub.groupby('ym')['time'].apply(mean_inter_event).rename('inter_event_time')
    grp = grp.join(energy_s).join(inter_s)

    grp = grp.reindex(date_range, fill_value=0)
    grp['monthly_mean_mag'] = grp['monthly_mean_mag'].fillna(0)
    grp['monthly_max_mag']  = grp['monthly_max_mag'].fillna(0)
    grp['inter_event_time'] = grp['inter_event_time'].fillna(0)
    grp.index = grp.index.to_timestamp()

    # Rolling features (pada skala asli)
    grp['rolling_mean_3'] = grp['monthly_count'].rolling(3, min_periods=1).mean().fillna(0)
    grp['rolling_mean_6'] = grp['monthly_count'].rolling(6, min_periods=1).mean().fillna(0)
    grp['rolling_std_3']  = grp['monthly_count'].rolling(3, min_periods=1).std().fillna(0)

    # Cumulative energy (normalized 0-1 pada skala asli)
    cum_e = grp['monthly_energy'].cumsum()
    max_e = cum_e.max()
    grp['cumulative_energy'] = (cum_e / max_e).fillna(0) if max_e > 0 else cum_e.fillna(0)

    # Lag features (pada skala asli)
    grp['lag_1'] = grp['monthly_count'].shift(1).fillna(0)
    grp['lag_2'] = grp['monthly_count'].shift(2).fillna(0)
    grp['lag_3'] = grp['monthly_count'].shift(3).fillna(0)

    # mag_anomaly
    roll_mag_12 = grp['monthly_mean_mag'].rolling(12, min_periods=1).mean()
    grp['mag_anomaly'] = (grp['monthly_mean_mag'] - roll_mag_12).fillna(0)

    raw = grp[FEATURES].copy()
    ts_all[z] = raw  # simpan skala asli

    # === LOG1p TRANSFORM pada kolom terpilih ===
    log_grp = raw.copy()
    for col in LOG_COLS:
        log_grp[col] = np.log1p(log_grp[col])
    ts_log[z] = log_grp

# Summary
print(f'Fitur input  : {N_FEATURES}')
print(f'Log1p cols   : {LOG_COLS}')
print()
print(f'{"Zona":<25} {"Months":>7} {"Active":>8} {"MaxCnt":>8} {"MeanCnt":>9}')
print('-' * 62)
for z in ZONES:
    ts  = ts_all[z]
    act = (ts['monthly_count'] > 0).sum()
    print(f'  {z:<23} {len(ts):>7} {act:>8} '
          f'{ts["monthly_count"].max():>8.0f} {ts["monthly_count"].mean():>9.2f}')

# Plot time series (skala asli)
fig, axes = plt.subplots(5, 1, figsize=(15, 14), sharex=True)
fig.suptitle('Monthly Earthquake Count per Zona (1950-2026)', fontsize=13, fontweight='bold')
for i, z in enumerate(ZONES):
    ax   = axes[i]
    ts   = ts_all[z]
    roll = ts['monthly_count'].rolling(12, min_periods=6).mean()
    ax.fill_between(ts.index, ts['monthly_count'], alpha=0.4, color=COLORS[z])
    ax.plot(ts.index, ts['monthly_count'], color=COLORS[z], lw=0.4, alpha=0.5)
    ax.plot(ts.index, roll, color='black', lw=1.3, alpha=0.85, label='12-mo MA')
    ax.set_ylabel('Count/mo', fontsize=8)
    ax.set_title(z, fontsize=9, color=COLORS[z], fontweight='bold', loc='left')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.2)
axes[-1].set_xlabel('Year')
plt.tight_layout()
plt.savefig(OUTDIR / 'timeseries_per_zona_v3.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] timeseries_per_zona_v3.png')


## Cell 3 — Preprocessing LSTM (Window=12, Scaler pada log-space)

In [ ]:
WINDOW    = 12
TRAIN_END = pd.Timestamp('2019-12-01')
VAL_END   = pd.Timestamp('2022-12-01')
N_F       = N_FEATURES  # 13

def make_sequences(data, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i : i + window])
        y.append(data[i + window, 0])  # col 0 = log1p(monthly_count)
    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.float32))

lstm_data = {}

print(f'Window = {WINDOW} bulan  |  Fitur = {N_F}  |  Target = log1p(monthly_count)')
print(f'Train: 1950-01 - 2019-12  |  Val: 2020-01 - 2022-12  |  Test: 2023-01 - 2026-05')
print()
print(f'{"Zona":<25} {"X_train":>10} {"X_val":>8} {"X_test":>8}  Shape')
print('-' * 68)

for z in ZONES:
    ts = ts_log[z]  # pakai versi log-transformed

    tr_mask = ts.index <= TRAIN_END
    va_mask = (ts.index > TRAIN_END) & (ts.index <= VAL_END)
    te_mask = ts.index > VAL_END

    tr = ts[tr_mask].values.astype(np.float32)
    va = ts[va_mask].values.astype(np.float32)
    te = ts[te_mask].values.astype(np.float32)

    scaler = MinMaxScaler((0, 1))
    tr_s   = scaler.fit_transform(tr)
    va_s   = scaler.transform(va)
    te_s   = scaler.transform(te)

    X_tr, y_tr = make_sequences(tr_s, WINDOW)
    X_va, y_va = make_sequences(np.vstack([tr_s[-WINDOW:], va_s]), WINDOW)
    X_te, y_te = make_sequences(np.vstack([va_s[-WINDOW:], te_s]), WINDOW)

    lstm_data[z] = {
        'scaler':       scaler,
        'X_train': X_tr, 'y_train': y_tr,
        'X_val':   X_va, 'y_val':   y_va,
        'X_test':  X_te, 'y_test':  y_te,
        'train_dates': ts[tr_mask].index[WINDOW:],
        'val_dates':   ts[va_mask].index,
        'test_dates':  ts[te_mask].index,
        'train_scaled': tr_s,
        'val_scaled':   va_s,
        'test_scaled':  te_s,
    }

    print(f'  {z:<23} {X_tr.shape[0]:>10,} {X_va.shape[0]:>8,} '
          f'{X_te.shape[0]:>8,}  {X_tr.shape}')

print(f'\nInput shape per sample : ({WINDOW}, {N_F})')
print('[OK] Preprocessing v3 selesai — scaler di log-space')


## Cell 4 — Arsitektur Model LSTM Simpel v3

In [ ]:
def build_lstm_v3(window=WINDOW, n_features=N_F):
    inp = keras.Input(shape=(window, n_features), name='input')
    x   = layers.LSTM(64, return_sequences=True,  name='lstm1')(inp)
    x   = layers.Dropout(0.3,                      name='drop1')(x)
    x   = layers.LSTM(32, return_sequences=False,  name='lstm2')(x)
    x   = layers.Dropout(0.2,                      name='drop2')(x)
    x   = layers.Dense(16, activation='relu',      name='dense1')(x)
    out = layers.Dense(1,  activation='linear',    name='output')(x)

    model = keras.Model(inputs=inp, outputs=out, name='LSTM_v3')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae'],
    )
    return model

_demo = build_lstm_v3()
_demo.summary()
print(f'\nInput  : ({WINDOW}, {N_F})')
print('Arch   : LSTM(64) -> Dropout(0.3) -> LSTM(32) -> Dropout(0.2) -> Dense(16) -> Dense(1)')
print('Loss   : MSE  |  Optimizer : Adam(lr=0.001)')
print('Target : log1p(monthly_count)  [inverse: expm1]')
del _demo


## Cell 5 — Training per Zona (epochs=300, batch=16, patience=20)

In [ ]:
import time

histories      = {}
trained_models = {}

fig, axes = plt.subplots(len(ZONES), 1, figsize=(12, 4 * len(ZONES)))
fig.suptitle('Training vs Validation Loss per Zona — v3 (log scale)',
             fontsize=13, fontweight='bold')

for i, z in enumerate(ZONES):
    d      = lstm_data[z]
    t0     = time.time()
    model  = build_lstm_v3()

    zone_fn  = z.replace(' ', '_').replace('-', '_')
    ckpt_pth = str(MODELDIR / f'lstm_{zone_fn}_v3.keras')

    cbs = [
        EarlyStopping(monitor='val_loss', patience=20,
                      restore_best_weights=True, verbose=0),
        ModelCheckpoint(ckpt_pth, save_best_only=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=10, min_lr=1e-6, verbose=0),
    ]

    hist = model.fit(
        d['X_train'], d['y_train'],
        validation_data=(d['X_val'], d['y_val']),
        epochs=300,
        batch_size=16,
        callbacks=cbs,
        verbose=0,
    )

    histories[z]      = hist
    trained_models[z] = model
    elapsed = time.time() - t0
    n_ep    = len(hist.history['loss'])
    best_vl = min(hist.history['val_loss'])
    print(f'[{i+1}/5] {z:<25}  epochs={n_ep:3d}  '
          f'best_val_loss={best_vl:.6f}  ({elapsed:.1f}s)')

    ax = axes[i]
    ax.plot(hist.history['loss'],     color=COLORS[z], lw=1.5, label='Train')
    ax.plot(hist.history['val_loss'], color=COLORS[z], lw=1.5, ls='--',
            alpha=0.75, label='Val')
    ax.set_title(z, fontsize=9, color=COLORS[z], fontweight='bold')
    ax.set_ylabel('MSE Loss')
    ax.set_yscale('log')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Epoch')
plt.tight_layout(rect=[0, 0, 1, 0.97])
out_path = OUTDIR / 'loss_per_zona_v3.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[OK] Saved: {out_path.name}')


## Cell 6 — Evaluasi + Diagnostic Plots (skala ASLI setelah expm1)

In [ ]:
def inv_count_v3(arr_sc, scaler, n_feat=N_F):
    """
    Inverse pipeline:
      scaled  -> MinMaxScaler.inverse  -> log1p(count)
      log1p(count) -> expm1            -> original count
    """
    dummy        = np.zeros((len(arr_sc), n_feat), dtype=np.float32)
    dummy[:, 0]  = arr_sc
    log_vals     = scaler.inverse_transform(dummy)[:, 0]
    return np.maximum(np.expm1(log_vals), 0.0)

# Referensi v1 & v2
R2_PREV = {
    'Zona Sumatera':      {'v1': -0.997,  'v2': -1.9777},
    'Zona Jawa-Bali-NTB': {'v1':  0.318,  'v2': -0.0266},
    'Zona Sulawesi-NTT':  {'v1':  0.159,  'v2': -0.0944},
    'Zona Maluku':        {'v1': -0.017,  'v2': -0.0379},
    'Zona Papua':         {'v1': -0.110,  'v2': -0.0162},
}

metrics_all = {}
pred_all    = {}

# --- Plot 1: Prediksi vs Aktual ---
fig1, axes1 = plt.subplots(len(ZONES), 1, figsize=(15, 3.5 * len(ZONES)))
fig1.suptitle('Prediksi vs Aktual — Test Set Jan 2023 - Mei 2026 (v3)',
              fontsize=13, fontweight='bold')

for i, z in enumerate(ZONES):
    d      = lstm_data[z]
    sc     = d['scaler']
    model  = trained_models[z]
    dates  = d['test_dates']

    y_pred_sc = model.predict(d['X_test'], verbose=0).flatten()
    y_pred    = inv_count_v3(y_pred_sc, sc)
    y_actual  = inv_count_v3(d['y_test'], sc)

    rmse = float(np.sqrt(mean_squared_error(y_actual, y_pred)))
    mae  = float(mean_absolute_error(y_actual, y_pred))
    nz   = y_actual > 0
    mape = (float(np.mean(np.abs(y_actual[nz] - y_pred[nz]) / y_actual[nz]) * 100)
            if nz.sum() > 0 else np.nan)
    r2   = float(r2_score(y_actual, y_pred))

    if len(y_actual) > 1 and np.std(y_pred) > 0:
        pearson_val, _ = pearsonr(y_actual, y_pred)
    else:
        pearson_val = np.nan

    metrics_all[z] = {
        'rmse':    round(rmse, 4),
        'mae':     round(mae, 4),
        'mape':    round(mape, 2) if not np.isnan(mape) else np.nan,
        'r2':      round(r2, 4),
        'pearson': round(float(pearson_val), 4) if not np.isnan(pearson_val) else np.nan,
    }
    pred_all[z] = {'dates': dates, 'actual': y_actual, 'pred': y_pred}

    ax = axes1[i]
    ax.plot(dates, y_actual, color='black',   lw=1.5, alpha=0.85, label='Aktual')
    ax.plot(dates, y_pred,   color=COLORS[z], lw=1.5, ls='--',    label='Prediksi v3')
    mape_s = f'{mape:.1f}%' if not np.isnan(mape) else 'N/A'
    ps_s   = f'{pearson_val:.3f}' if not np.isnan(pearson_val) else 'N/A'
    ax.set_title(
        f'{z}  |  RMSE={rmse:.2f}  MAE={mae:.2f}  MAPE={mape_s}  '
        f'R2={r2:.3f}  Pearson={ps_s}',
        fontsize=8.5, color=COLORS[z])
    ax.set_ylabel('Count / mo')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

axes1[-1].set_xlabel('Month')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(OUTDIR / 'prediksi_vs_aktual_v3.png', dpi=300, bbox_inches='tight')
plt.show()
print('[OK] prediksi_vs_aktual_v3.png')

# --- Plot 2: Residual Analysis (2 kolom × 5 baris) ---
fig2, axes2 = plt.subplots(len(ZONES), 2, figsize=(14, 3.2 * len(ZONES)))
fig2.suptitle('Residual Analysis per Zona — v3', fontsize=13, fontweight='bold')

for i, z in enumerate(ZONES):
    p      = pred_all[z]
    y_act  = p['actual']
    y_pr   = p['pred']
    resid  = y_act - y_pr

    # kiri: scatter actual vs predicted
    ax_s = axes2[i, 0]
    ax_s.scatter(y_act, y_pr, color=COLORS[z], alpha=0.6, s=25, edgecolors='none')
    lims = [min(y_act.min(), y_pr.min()) * 0.95,
            max(y_act.max(), y_pr.max()) * 1.05]
    ax_s.plot(lims, lims, 'k--', lw=1, alpha=0.5, label='Perfect')
    ax_s.set_xlim(lims)
    ax_s.set_ylim(lims)
    ax_s.set_xlabel('Aktual')
    ax_s.set_ylabel('Prediksi')
    ax_s.set_title(f'{z} — Scatter', fontsize=8, color=COLORS[z], fontweight='bold')
    r2_s = metrics_all[z]['r2']
    ax_s.legend(fontsize=7)
    ax_s.text(0.05, 0.92, f'R2={r2_s:.3f}', transform=ax_s.transAxes,
              fontsize=8, color=COLORS[z], fontweight='bold')
    ax_s.grid(alpha=0.25)

    # kanan: distribusi residual
    ax_h = axes2[i, 1]
    ax_h.hist(resid, bins=15, color=COLORS[z], alpha=0.7, edgecolor='white')
    ax_h.axvline(0,           color='black', lw=1.5, ls='-',  label='Zero')
    ax_h.axvline(resid.mean(), color='red',   lw=1.2, ls='--', label=f'Mean={resid.mean():.1f}')
    ax_h.set_xlabel('Residual (Aktual - Prediksi)')
    ax_h.set_ylabel('Frekuensi')
    ax_h.set_title(f'{z} — Distribusi Residual', fontsize=8, color=COLORS[z], fontweight='bold')
    ax_h.legend(fontsize=7)
    ax_h.grid(alpha=0.25)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(OUTDIR / 'residual_analysis_v3.png', dpi=300, bbox_inches='tight')
plt.show()
print('[OK] residual_analysis_v3.png')

# --- Tabel metrik + perbandingan ---
sep = '=' * 105
print(f'\n{sep}')
print('METRIK EVALUASI TEST SET — v3 vs v2 vs v1')
print(sep)
delta_hdr = 'dR2(v1)'
hdr = (f'{"Zona":<25} {"RMSE":>8} {"MAE":>8} {"MAPE%":>8} '
       f'{"R2_v3":>8} {"R2_v2":>8} {"R2_v1":>8} {delta_hdr:>9} '
       f'{"Pearson":>8} {"Status":>10}')
print(hdr)
print('-' * 105)
for z in ZONES:
    m    = metrics_all[z]
    r2v1 = R2_PREV[z]['v1']
    r2v2 = R2_PREV[z]['v2']
    r2v3 = m['r2']
    dv1  = r2v3 - r2v1
    ms   = f'{m["mape"]:>8.2f}' if not np.isnan(m['mape']) else '     N/A'
    ps   = f'{m["pearson"]:>8.4f}' if not np.isnan(m['pearson']) else '     N/A'
    sign = '+' if dv1 >= 0 else ''
    if r2v3 >= 0.5:
        status = 'EXCELLENT'
    elif r2v3 >= 0.3:
        status = 'GOOD'
    elif r2v3 >= 0:
        status = 'OK'
    elif r2v3 >= -0.1:
        status = 'MARGINAL'
    else:
        status = 'CHECK'
    print(f'  {z:<23} {m["rmse"]:>8.3f} {m["mae"]:>8.3f} {ms} '
          f'{r2v3:>8.4f} {r2v2:>8.4f} {r2v1:>8.3f} {sign}{dv1:>8.3f} '
          f'{ps} {status:>10}')
print(sep)


## Cell 7 — Forecast 12 Bulan (Jun 2026 – Mei 2027)

In [ ]:
FORECAST_N     = 12
FORECAST_START = pd.Timestamp('2026-06-01')
forecast_dates = pd.date_range(FORECAST_START, periods=FORECAST_N, freq='MS')

forecast_all         = {}
gap_periods_forecast = {}

fig, axes = plt.subplots(len(ZONES), 1, figsize=(14, 4 * len(ZONES)))
fig.suptitle('Forecast Seismisitas Jun 2026 - Mei 2027 — v3',
             fontsize=13, fontweight='bold')

for i, z in enumerate(ZONES):
    d      = lstm_data[z]
    sc     = d['scaler']
    model  = trained_models[z]

    # CI dari residual test set (skala asli)
    ci_margin = 1.96 * float(np.std(
        pred_all[z]['actual'] - pred_all[z]['pred']))

    # Seed: 12 bulan terakhir data (scaled log-space)
    all_sc = np.vstack([d['train_scaled'], d['val_scaled'], d['test_scaled']])
    seed   = all_sc[-WINDOW:].copy()  # (12, 13)

    # Rata-rata fitur non-count dari bulan aktif
    nz_m       = all_sc[:, 0] > 0
    hist_other = (all_sc[nz_m, 1:].mean(axis=0) if nz_m.sum() > 0
                  else all_sc[:, 1:].mean(axis=0))

    # Rolling forecast
    fc_sc   = []
    win_now = seed.copy()
    for _ in range(FORECAST_N):
        x_in   = win_now[np.newaxis]
        pred_s = float(np.clip(model.predict(x_in, verbose=0)[0, 0], 0, 1))
        fc_sc.append(pred_s)
        new_row    = np.zeros(N_F, dtype=np.float32)
        new_row[0] = pred_s
        new_row[1:] = hist_other
        win_now = np.vstack([win_now[1:], new_row[np.newaxis]])

    # Inverse: scaled -> log1p -> expm1 -> count
    fc_arr  = np.array(fc_sc, dtype=np.float32)
    dummy_f = np.zeros((FORECAST_N, N_F), dtype=np.float32)
    dummy_f[:, 0] = fc_arr
    log_vals = sc.inverse_transform(dummy_f)[:, 0]
    fc_cnt   = np.maximum(np.expm1(log_vals), 0.0)

    ci_lo = np.maximum(fc_cnt - ci_margin, 0.0)
    ci_hi = fc_cnt + ci_margin

    # Gap detection (dibandingkan skala asli train)
    tr_cnt  = ts_all[z].loc[ts_all[z].index <= TRAIN_END, 'monthly_count']
    gap_thr = float(max(0.0, tr_cnt.mean() - 1.5 * tr_cnt.std()))
    is_gap  = fc_cnt < gap_thr

    forecast_all[z] = {
        'dates': forecast_dates, 'forecast': fc_cnt,
        'ci_lower': ci_lo, 'ci_upper': ci_hi,
        'is_gap': is_gap, 'gap_threshold': gap_thr,
    }
    gap_periods_forecast[z] = bool(is_gap.any())

    ax     = axes[i]
    hist24 = ts_all[z]['monthly_count'].iloc[-24:]
    ax.plot(hist24.index, hist24.values,
            color='gray', lw=1.2, alpha=0.7, label='Historis 24 bln')
    ax.plot(forecast_dates, fc_cnt,
            color=COLORS[z], lw=2, marker='o', ms=4, label='Forecast v3')
    ax.fill_between(forecast_dates, ci_lo, ci_hi,
                    color=COLORS[z], alpha=0.2, label='CI 95%')
    gap_idx = np.where(is_gap)[0]
    if len(gap_idx):
        ax.scatter(forecast_dates[gap_idx], fc_cnt[gap_idx],
                   color='red', zorder=5, s=70, marker='v', label='Gap period')
    ax.axhline(gap_thr, color='red', ls=':', lw=1, alpha=0.6,
               label=f'Gap thr={gap_thr:.1f}')
    ax.axvline(FORECAST_START, color='black', ls='--', lw=0.8, alpha=0.4)
    ax.set_title(f'{z}  |  Gap: {is_gap.sum()}/12',
                 fontsize=9, color=COLORS[z], fontweight='bold')
    ax.set_ylabel('Count / mo')
    ax.legend(fontsize=7, ncol=3)
    ax.grid(alpha=0.25)

axes[-1].set_xlabel('Month')
plt.tight_layout(rect=[0, 0, 1, 0.97])
out_path = OUTDIR / 'forecast_2026_2027_v3.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[OK] {out_path.name}')

print(f'\n{"Zona":<25} {"Gap Mo":>7} {"Min FC":>9} {"Thr":>10}')
print('-' * 56)
for z in ZONES:
    f = forecast_all[z]
    print(f'  {z:<23} {f["is_gap"].sum():>7} '
          f'{f["forecast"].min():>9.1f} {f["gap_threshold"]:>10.1f}')


## Cell 8 — Simpan Semua Output & Diagnostik Zona R² < 0

In [ ]:
# 1. Predictions per zona v3
for z in ZONES:
    p   = pred_all[z]
    zfn = z.replace(' ', '_').replace('-', '_')
    pd.DataFrame({
        'month':     [d.strftime('%Y-%m') for d in p['dates']],
        'actual':    p['actual'].round(4).tolist(),
        'predicted': p['pred'].round(4).tolist(),
        'residual':  (p['actual'] - p['pred']).round(4).tolist(),
    }).to_csv(OUTDIR / f'predictions_{zfn}_v3.csv', index=False)

# 2. Forecast CSV v3
fc_rows = []
for z in ZONES:
    f = forecast_all[z]
    for j, d in enumerate(f['dates']):
        fc_rows.append({
            'month':         d.strftime('%Y-%m'),
            'zone_name':     z,
            'forecast':      round(float(f['forecast'][j]), 4),
            'ci_lower':      round(float(f['ci_lower'][j]), 4),
            'ci_upper':      round(float(f['ci_upper'][j]), 4),
            'is_gap_period': bool(f['is_gap'][j]),
        })
fc_df = pd.DataFrame(fc_rows)
fc_df.to_csv(WORKDIR / 'lstm_forecast_2026_2027_v3.csv', index=False)

# 3. Metrics CSV v3
mt_df = pd.DataFrame([{'zone_name': z, **metrics_all[z]} for z in ZONES])
mt_df.to_csv(WORKDIR / 'lstm_metrics_v3.csv', index=False)

# Konfirmasi output
sep72 = '=' * 72
print(sep72)
print('KONFIRMASI OUTPUT v3')
print(sep72)
check_files = (
    [OUTDIR / f for f in ['timeseries_per_zona_v3.png', 'loss_per_zona_v3.png',
                           'prediksi_vs_aktual_v3.png', 'residual_analysis_v3.png',
                           'forecast_2026_2027_v3.png']]
    + [OUTDIR / f'predictions_{z.replace(" ","_").replace("-","_")}_v3.csv'
       for z in ZONES]
    + [WORKDIR / 'lstm_metrics_v3.csv',
       WORKDIR / 'lstm_forecast_2026_2027_v3.csv']
    + [MODELDIR / f'lstm_{z.replace(" ","_").replace("-","_")}_v3.keras'
       for z in ZONES]
)
all_ok = True
for fp in check_files:
    if fp.exists():
        print(f'  [OK]  {fp.name:<55} {fp.stat().st_size/1024:>7.1f} KB')
    else:
        print(f'  [MISSING] {fp}')
        all_ok = False

print(f'\n{sep72}')
print('SEMUA OUTPUT TERSIMPAN' if all_ok else 'ADA FILE MISSING')
print(sep72)

# Metrics summary
print(f'\nLSTM METRICS v3:')
print(mt_df.to_string(index=False))

# ===== DIAGNOSTIK KHUSUS ZONA R² < 0 =====
problem_zones = [z for z in ZONES if metrics_all[z]['r2'] < 0]

if problem_zones:
    print(f'\n{sep72}')
    print('DIAGNOSTIK ZONA R2 < 0')
    print(sep72)

    for z in problem_zones:
        m    = metrics_all[z]
        p    = pred_all[z]
        ts   = ts_all[z]
        resid = p['actual'] - p['pred']

        print(f'\n--- {z} ---')
        print(f'  R2      = {m["r2"]:.4f}  (negatif = model < mean baseline)')
        print(f'  RMSE    = {m["rmse"]:.2f}')
        print(f'  MAE     = {m["mae"]:.2f}')
        print(f'  Pearson = {m["pearson"]}')
        print(f'  Residual mean  = {resid.mean():.2f} (bias sistematis)')
        print(f'  Residual std   = {resid.std():.2f}')
        print(f'  Actual  range  = [{p["actual"].min():.1f}, {p["actual"].max():.1f}]')
        print(f'  Pred    range  = [{p["pred"].min():.1f}, {p["pred"].max():.1f}]')

        # Hitung statistik zona
        te_mask  = ts.index > VAL_END
        act_mean = p['actual'].mean()
        act_std  = p['actual'].std()
        cv       = act_std / act_mean * 100 if act_mean > 0 else 0

        print(f'  CV (coeff var) = {cv:.1f}%  ('
              f'{"HIGH variability" if cv > 80 else "moderate"}'  f')')

        # Kemungkinan penyebab
        causes = []
        if cv > 100:
            causes.append('Variabilitas sangat tinggi — seismisitas episodik, sulit diprediksi')
        if p['actual'].max() > 3 * act_mean:
            causes.append('Spike ekstrem — kemungkinan aftershock sequence mendominasi')
        if abs(resid.mean()) > 0.5 * act_mean:
            causes.append('Bias sistematis besar — model under/over-estimate secara konsisten')
        if m['pearson'] < 0.2 if not np.isnan(m['pearson']) else True:
            causes.append('Pearson rendah — model tidak menangkap tren temporal zona ini')
        if not causes:
            causes.append('R2 negatif tapi margin kecil — model mendekati mean baseline')

        print(f'  Kemungkinan penyebab:')
        for c in causes:
            print(f'    - {c}')

        print(f'  Rekomendasi:')
        print(f'    - Coba seasonal decomposition (STL) sebelum LSTM')
        print(f'    - Tambah fitur tektonik/geofisik (jarak sesar, b-value lokal)')
        print(f'    - Pertimbangkan model hybrid (LSTM + XGBoost residual correction)')
        print(f'    - Untuk zona episodik, gunakan classification (high/low activity)')

else:
    print(f'\n[OK] Semua zona memiliki R2 >= 0. Target tercapai.')

# Rangkuman perbandingan v1/v2/v3
print(f'\n{sep72}')
print('RANGKUMAN R2: v1 -> v2 -> v3')
print(sep72)
r2_col = 'R2'
print(f'{"Zona":<25} {"R2_v1":>8} {"R2_v2":>8} {"R2_v3":>8} {"Trend":>8}')
print('-' * 58)
for z in ZONES:
    r1 = R2_PREV[z]['v1']
    r2 = R2_PREV[z]['v2']
    r3 = metrics_all[z]['r2']
    if r3 > r1 and r3 > r2:
        trend = 'BEST'
    elif r3 > r1:
        trend = '^v1'
    elif r3 > r2:
        trend = '^v2'
    else:
        trend = 'check'
    print(f'  {z:<23} {r1:>8.3f} {r2:>8.4f} {r3:>8.4f} {trend:>8}')
print(sep72)
